---
title: Skysat Processing Workflow
description: This notebook demonstrates the complete workflow for downloading and processing Skysat satellite imagery using the `disasters-product-algorithms` package. 
author: 
  - Ethan Kerr (Editor, UAH)
  - Kyle Lesinger (Editor, UAH)
date: September 2, 2026
execute:
   freeze: true
---

# Run This Notebook

<div class="alert alert-block" style="
     background-color: #f8d7da;
     color: #721c24;
     border-left: 4px solid #28a745;
  ">
Disclaimer: it is highly recommended to run a tutorial within NASA VEDA JupyterHub, which already includes functions for processing and visualizing data specific to VEDA stories. Running the tutorial outside of the VEDA JupyterHub may lead to errors, specifically related to EarthData authentication. Additionally, it is recommended to use the Pangeo workspace within the VEDA JupyterHub, since certain packages relevant to this tutorial are already installed. </div>

<h4> If you <strong>do not</strong> have a VEDA Jupyterhub Account you can launch this notebook on your local environment using MyBinder by clicking the icon below.</h4>
<br/>
<a href="https://binder.openveda.cloud/v2/gh/NASA-IMPACT/veda-docs/9c8cdbae92906fb7062b8a0c759dad90e223a4f9?urlpath=lab%2Ftree%2Fuser-guide%2Fnotebooks%2Fstories%2Fderechos.ipynb">
<img src="https://binder.openveda.cloud/badge_logo.svg" alt="Binder" title="A cute binder" width="150"/> </a>

## Table of Contents
- [Skysat Processing Workflow](#skysat-processing-workflow)
- [Environment Setup](#environment-setup)
- [Process Skysat Data](#process-skysat-data)
- [View Results](#view-results)
- [Next Steps](#next-steps)

# Skysat Processing Workflow #

This notebook demonstrates the complete workflow for downloading and processing Skysat imagery using the `disasters-product-algorithms` package.

## Workflow Steps
1. **Configure Environment Variables** - Set processing parameters
2. **Process Skysat Data** - Generate products with COG conversion
3. **View Results** - Examine the generated outputs
4. **Upload** - Upload files to the S3 bucket

## Features Demonstrated
- Cloud Optimized GeoTIFF (COG) conversion
- Multiple product generation (true color, color IR, NDVI, NDWI, EVI)
- Gamma correction (color composites)

# Environment Setup #

Configure all processing parameters as environment variables for easy modification.

You can choose to apply a filter to the RGB imagery to enhance the colors and contrast of the image with the ```RGB_ENHANCEMENT``` toggle. There are two processing steps to this enhancer. The first step is a per-band normalization by stretching the data from the 2nd percentile to the 98th percentile pixel reflectances. The pixels are normalized on a 0-1 scale. The second step is applying a Gamma correction by adjusting the ```GAMMA``` variable. The Gamma correction is an equation applied to each pixel:
$$pixel_{enhance} = pixel^{1/GAMMA}$$
A standard value for ```GAMMA``` is 2.2. This enhances the image by brightening dark regions. A power less than zero (1/2.2) will increase the reflectance of pixels with normalized reflectances closer to zero greater than pixels closer to one. Then, the pixel values are converted back to an 8-bit (0-255) value for proper RGB visualization. However, for SkySat data, the processed images are already well-lit and contrasted, so a lower gamma value might be more appropriate. The default ```GAMMA``` is 1.0, which doesn't change the pixels.

In [ ]:
import sys
import os

# Tell Python to look in the 'src' folder for shared_utils
sys.path.append("../src")

# ==============================================================================
# ACTIVATION OPTIONS BLOCK
# Change these variables for each new disaster activation.
# ==============================================================================

# Metadata
EVENT_NAME = "202406_Example_Event"
SOURCE = "CSDA"

# Data selection
DATE = "2026-07-14 12:00:00"

# Asset used for the COLOR COMPOSITES: "visual" (true color only), "basic_analytic"
# (composites only; RPC-orthorectified) or "analytic". Indices (ndvi/ndwi/evi)
# ALWAYS read the 4-band analytic asset -- it is the only one carrying NIR.
SKYSAT_PRODUCT_TYPE = "analytic"

# Product flags
PRODUCTS = {
    "truecolor": False,
    "colorir": False,
    "ndvi": False,
    "ndwi": True,
    "evi": False,
}

# RGB enhancement
GAMMA = 1.0

# Output settings
OUTPUT_DIR = "/tmp/skysat_output"

# S3 upload settings
ENABLE_S3_UPLOAD = False

# Destination. The bucket and every product directory are hard-coded once, in
# shared_utils.product_paths; nothing here keeps its own copy. Products publish to
# s3://<bucket>/ProgramData/<Sensor>/<Product>/<filename>.tif -- no event and no
# date level in the key, because the activation lives in the GeoTIFF tags.
from shared_utils.product_paths import STAGING_BUCKET

S3_BUCKET = STAGING_BUCKET
PRODUCT_SENSOR = "skysat"       # key into shared_utils.product_paths

In [ ]:
import subprocess
import json, tempfile
from pathlib import Path
from shared_utils import PROCESSOR_STRING
from shared_utils.s3utils import retrieve_s3_valid_dates
from pprint import pprint

# 1. Setup Environment
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Print Configuration
print("Configuration:")
print(f"  Date: {DATE}")
print(f"  Source: {SOURCE}")
print(f"  Products: {PRODUCTS}")
print(f"  Output Directory: {OUTPUT_DIR}")

# 3. Create Metadata File
ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
}
_meta_fd, ACTIVATION_METADATA_PATH = tempfile.mkstemp(
    prefix='activation_meta_', suffix='.json'
)
with os.fdopen(_meta_fd, 'w') as _f:
    json.dump(ACTIVATION_METADATA, _f)

# 4. Diagnostic Check
_avail_bucket = "csdap-planet-skysat-delivery"
_avail_prefix = "disasters"
#available_dates = retrieve_s3_valid_dates(_avail_bucket, _avail_prefix)
#print(f"{len(available_dates)} capture dates available in s3://{_avail_bucket}/{_avail_prefix}/ at LEVEL={LEVEL}")
#pprint(available_dates)

# Process Skysat Data #

Process the downloaded imagery to generate various products with COG conversion and event naming. A given datetime can be broken into several scenes, so multiple output files may be generated for even one product selection.

**Note:** The processing script has been configured to display progress in real-time within JupyterHub. You'll see:
- Detailed product generation steps
- COG conversion progress
- Error messages if any products fail
- Final processing summary with success/failure counts
- Log file location for detailed error tracking

**Important:** Depending on how many scenes are in your requested datetime, processing will take at least several minutes.

In [ ]:
process_cmd = [
    "process_skysat",
    "-h"
]

help_flags = subprocess.run(process_cmd, cwd=os.getcwd())
print(help_flags)

In [ ]:
# Dispatch one process_skysat subprocess per enabled product.
#
# max_workers=1 (SERIAL) because the binding constraint here is RAM, not CPU.
# Each subprocess holds the whole scene in memory several times over.
#
# COG settings (NoData, native CRS, compression and compression level) are fixed inside
# the SkySat CLI and are not passed as command-line arguments.

from shared_utils.parallel import map_threaded

INDEX_PRODUCTS = {"ndvi", "ndwi", "evi"}

cmds = []
for PRODUCT, ENABLED in PRODUCTS.items():
    if not ENABLED:
        continue

    cmd = [
        "process_skysat",
        "--product", PRODUCT,
        "--date", DATE,
        "--output", OUTPUT_DIR,
    ]

    # Gamma correction applies to color composites only.
    if PRODUCT in ["truecolor", "colorir"]:
        cmd.extend(["--product-type", SKYSAT_PRODUCT_TYPE, "--gamma", str(GAMMA)])

    cmd.extend(["--metadata-json", ACTIVATION_METADATA_PATH])
    cmds.append((PRODUCT, cmd))


def _run(item):
    product, cmd = item
    print(f"\nProcessing SkySat product: {product}")
    print(f"Command: {' '.join(cmd)}\n")

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    print(f"--- {product} stdout ---\n{result.stdout}")

    if result.stderr:
        print(f"--- {product} stderr ---\n{result.stderr}")

    return product, result.returncode


results = map_threaded(
    _run,
    cmds,
    max_workers=1,
    desc="SkySat products",
)

for r in results:
    if isinstance(r, Exception):
        print(f"\n✗ SkySat processing crashed: {r}\n")
    else:
        product, return_code = r

        if return_code == 0:
            print(
                f"\n✓ SkySat {product} processing completed successfully!\n"
            )
        else:
            print(
                f"\n✗ SkySat {product} failed with return code "
                f"{return_code}\n"
            )

# View Results #

Examine the generated output files and directory structure.

In [ ]:
import glob
import rasterio
import matplotlib.pyplot as plt
import numpy as np

output_dir = os.path.abspath(OUTPUT_DIR)

tif_files = sorted([
    f for f in glob.glob(os.path.join(output_dir, "**/*.tif"), recursive=True)
    if not f.endswith(".tmp.tif")
])

print(f"\nFound {len(tif_files)} outputs\n")

for tif_file in tif_files:

    if 'skysat' in tif_file.lower():
        print(f"\nDisplaying: {os.path.basename(tif_file)}")
    
        with rasterio.open(tif_file) as src:
    
            plt.figure(figsize=(10, 10))
    
            filename = os.path.basename(tif_file).lower()
    
            # RGB products (composite images)
            if src.count >= 3:
    
                arr = src.read([1, 2, 3]).astype(float)
                arr = np.transpose(arr, (1, 2, 0))
    
                if "truecolor" in filename or "colorir" in filename:
                    arr = np.clip(arr / 255.0, 0, 1)
    
                else:
                    band_min = np.min(arr, axis=(0, 1))
                    band_max = np.max(arr, axis=(0, 1))
                    arr = (arr - band_min) / (band_max - band_min + 1e-6)
                    arr = np.clip(arr, 0, 1)
    
                # Composites carry a 4th ALPHA band (0 = nodata fill) instead of
                # a scalar nodata -- 0 is a legitimate 8-bit sample. Stack it in
                # so the fill border renders transparent rather than black.
                if src.count >= 4:
                    alpha = src.read(4).astype(float) / 255.0
                    arr = np.dstack([arr, alpha])
                    print(f"  alpha band present: {(alpha == 0).sum()} nodata px")
    
                plt.imshow(arr)
                plt.title(filename)
                plt.axis("off")
                plt.show()
    
            # Single band products (indices)
            # You can adjust vmin and vmax for each index for visualization to increase image contrast
            else:
                arr = src.read(1).astype(float)
    
                if "ndvi" in filename:
                    cmap = "RdYlGn"
                    arr = np.where((arr < -1) | (arr > 1), np.nan, arr)
                    im = plt.imshow(arr, cmap=cmap, vmin=-0.5, vmax=0.5)
    
                elif "ndwi" in filename:
                    cmap = "Blues"
                    arr = np.where((arr < -1) | (arr > 1), np.nan, arr)
                    im = plt.imshow(arr, cmap=cmap)
    
                elif "evi" in filename:
                    cmap = "viridis"
                    arr = np.where((arr < -1) | (arr > 1), np.nan, arr)
                    im = plt.imshow(arr, cmap=cmap, vmin=-0.5, vmax=0.5)
    
                else:
                    im = plt.imshow(arr, cmap="viridis")
    
                plt.colorbar(im)
                plt.title(filename)
                plt.axis("off")
                plt.show()
    
        print(f"✓ Plotted {os.path.basename(tif_file)}")
    else:
        continue

# Interactive Visualization

Using the leafmap package, we can visualize a file on a map projection with pan and zoom capabilities. With this, we can see the high-resolution details of Skysat and view the geolocation of the file.

In [ ]:
! pip install leafmap
! pip install localtileserver

In [ ]:
import leafmap
m = leafmap.Map()
m.add_raster(tif_files[0], resampling_method = 'near', layer_name="Skysat") # adjust index in tif_files[_] to change file being viewed
m

# Next Steps #

You can now:
1. Load and visualize the GeoTIFF files using libraries like `rasterio` or `GDAL`
2. Upload the COG files to cloud storage (S3, GCS, etc.)
3. Process additional dates or tiles by modifying the configuration variables
4. Generate additional products by updating the `PRODUCTS` list

# Upload to S3 (Optional) #

Publish the finished COGs to S3. **Opt-in** — runs only when `ENABLE_S3_UPLOAD = True` in the configuration cell. Each COG is uploaded to `s3://{S3_BUCKET}/ProgramData/<Sensor>/<Product>/<filename>` via `shared_utils.upload_file_to_s3`, with the destination resolved from `shared_utils.product_paths` rather than a prefix set here. When merging, only the merged COGs are published.

In [ ]:
# ==================== UPLOAD TO S3 (optional) ====================
# Runs after processing. Set ENABLE_S3_UPLOAD = True in the config cell to publish
# the finished COGs to
# s3://{S3_BUCKET}/ProgramData/<Sensor>/<Product>/<filename>.
import glob, os
from shared_utils import upload_file_to_s3
from shared_utils.product_paths import prefix_for_product_dir

if not ENABLE_S3_UPLOAD:
    print("S3 upload OFF (set ENABLE_S3_UPLOAD = True in the config cell to publish).")
else:
    _base = OUTPUT_DIR
    cogs = [f for f in glob.glob(os.path.join(_base, "**", "*.tif"), recursive=True)
            if not f.endswith(".tmp.tif")]
    # When merging, publish ONLY the merged COGs (skip per-tile merge inputs).
    # globals().get(...) so notebooks without ENABLE_MERGE don't NameError.
    if globals().get("ENABLE_MERGE"):
        cogs = [f for f in cogs if "merged" in os.path.basename(f)]
    for f in cogs:
        # upload_file_to_s3 uses default AWS credentials. If AccessDenied, the bucket
        # needs the upload role -- swap to
        # shared_utils.s3_operations.initialize_s3_client() + upload_to_s3().
        #
        # The processor wrote each COG into its product directory, so the published
        # key is that directory resolved through the shared table. No local prefix
        # is involved, and no date or event level reaches the key.
        _prod = os.path.basename(os.path.dirname(f))
        _key = f"{prefix_for_product_dir(PRODUCT_SENSOR, _prod)}/{os.path.basename(f)}"
        upload_file_to_s3(f, f"s3://{S3_BUCKET}/{_key}")
    print(f"\nUploaded {len(cogs)} COG(s) to "
          f"s3://{S3_BUCKET}/ProgramData/<Sensor>/<Product>/")